# MT04 — Behavior Cloning with PyTorch

### Lab Description

**Behavior cloning (BC)** is the simplest form of imitation learning: collect `(observation, action)` pairs from an expert, then train a neural network to map observations to the expert's actions with plain supervised learning. It is how many vision-language-action and manipulation policies are bootstrapped.

Here the *expert* is the scripted reach from MT03. You will collect a dataset across many randomized cube positions, train a small PyTorch MLP on the AMD GPU to imitate it, and then let the **learned network** drive the arm — closing the loop from scripted behaviour to a learned policy.

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.4 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud you will have the ROCm + PyTorch environment (the `auplc-mujoco-torch` course image: torch + mujoco + robosuite + gymnasium) that this notebook is built for.

## Goals
- Collect an expert demonstration dataset in robosuite
- Build and train a PyTorch policy network on the AMD GPU
- Use mean-squared-error behavior cloning
- Roll out the learned policy and verify it reaches the cube

### Imports and renderer

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import numpy as np
import mujoco
import imageio
import robosuite as suite
from robosuite import load_composite_controller_config
from IPython.display import Video

import logging
logging.getLogger("robosuite_logs").setLevel(logging.ERROR)  # hide repetitive controller-config warnings

os.makedirs("output/videos", exist_ok=True)
print("robosuite", suite.__version__, "| mujoco", mujoco.__version__)

# robosuite's built-in camera-observation renderer can emit intermittently
# corrupted frames on this ROCm/EGL stack. We instead drive mujoco.Renderer
# directly (the reliable path) and show only the visual geom group (group 1),
# which avoids the green/blue collision shapes and looks correct.
def make_renderer(env, height=256, width=256):
    R = mujoco.Renderer(env.sim.model._model, height=height, width=width)
    opt = mujoco.MjvOption()
    opt.geomgroup[:] = 0
    opt.geomgroup[1] = 1
    return R, opt

def grab_frame(env, R, opt, camera="frontview"):
    mujoco.mj_forward(env.sim.model._model, env.sim.data._data)
    R.update_scene(env.sim.data._data, camera=camera, scene_option=opt)
    return R.render()

### Collect expert demonstrations

The expert is a task-space P-controller that drives the end-effector toward the cube (the MT03 reach). For each of many episodes the cube starts in a new random position, so the dataset covers many situations. We record the **state** (end-effector position + cube position, 6 numbers) and the **expert action** (7 numbers) at every step.

In [ ]:
import torch
import torch.nn as nn
device = "cuda" if torch.cuda.is_available() else "cpu"
print("training device:", device)

controller = load_composite_controller_config(controller="BASIC", robot="Panda")
env = suite.make(env_name="Lift", robots="Panda", controller_configs=controller,
                 has_renderer=False, has_offscreen_renderer=False, use_camera_obs=False,
                 control_freq=20, horizon=80)

def state_vec(obs):
    # compact state: end-effector position (3) + cube position (3)
    return np.concatenate([obs["robot0_eef_pos"], obs["cube_pos"]]).astype(np.float32)

def expert_action(obs):
    delta = np.clip((obs["cube_pos"] - obs["robot0_eef_pos"]) * 5, -1, 1)
    return np.array([delta[0], delta[1], delta[2], 0, 0, 0, -1.0], dtype=np.float32)

X, Y = [], []
for episode in range(40):
    obs = env.reset()
    for _ in range(60):
        X.append(state_vec(obs)); Y.append(expert_action(obs))
        obs, reward, done, info = env.step(expert_action(obs))
X = torch.tensor(np.array(X), device=device)
Y = torch.tensor(np.array(Y), device=device)
print("collected dataset:", X.shape[0], "state-action pairs  | state dim", X.shape[1], "action dim", Y.shape[1])

### Train the policy network (behavior cloning)

A small multilayer perceptron maps the 6-D state to the 7-D action. We minimize the mean-squared error between the network's output and the expert's action with Adam — ordinary supervised regression, run on the GPU. The loss should fall steadily.

In [ ]:
import matplotlib.pyplot as plt

policy = nn.Sequential(
    nn.Linear(X.shape[1], 128), nn.ReLU(),
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, Y.shape[1]),
).to(device)
optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)

losses = []
for epoch in range(400):
    optimizer.zero_grad()
    loss = ((policy(X) - Y) ** 2).mean()
    loss.backward(); optimizer.step()
    losses.append(loss.item())
print("final behavior-cloning loss:", round(losses[-1], 6))

plt.figure(figsize=(7, 3))
plt.plot(losses); plt.xlabel("epoch"); plt.ylabel("MSE loss")
plt.title("Behavior cloning training loss"); plt.yscale("log"); plt.grid(True); plt.show()

### Roll out the learned policy

Now the **network** chooses the actions: each step we feed the current state through `policy` and apply its output. If BC worked, the learned policy drives the end-effector to the cube without any hand-written control law. We measure the final end-effector-to-cube distance and render the rollout.

In [ ]:
R, opt = make_renderer(env, 256, 256)
obs = env.reset()
frames = [grab_frame(env, R, opt, camera="frontview")]
policy.eval()
for _ in range(80):
    s = torch.tensor(state_vec(obs), device=device).unsqueeze(0)
    with torch.no_grad():
        action = policy(s).squeeze(0).cpu().numpy()
    obs, reward, done, info = env.step(action)
    frames.append(grab_frame(env, R, opt, camera="frontview"))
R.close()
final_dist = float(np.linalg.norm(obs["cube_pos"] - obs["robot0_eef_pos"]))
print(f"learned policy: final end-effector-to-cube distance = {final_dist:.3f} m")
imageio.mimsave("output/videos/mt04_bc_policy.mp4", frames, fps=20)
print("saved", len(frames), "frames")

### Watch the learned policy

A small final distance means the cloned network learned to reach the cube purely from the expert demonstrations.

In [ ]:
Video(url="output/videos/mt04_bc_policy.mp4")

## Conclusions

You collected demonstrations, trained a PyTorch policy on the AMD GPU, and watched the **learned** network reach the cube — a complete behavior-cloning pipeline. This is the same recipe (just bigger networks, image inputs, and more data) behind modern imitation and vision-language-action policies. Together, MT01–04 take you from robosuite basics, through controllers and the RL interface, to a neural policy learned on AMD hardware. MT05 onward scales the learner from a small MLP to pretrained vision-language-action models.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT